# Sentiment classification with metaphor features

Run the configuration cell first. Set `DATASET_NAME` to `"Book"`, `"IMDB"`, or `"SST2"`.

- With `USE_PRECOMPUTED_LABELS = True` (default), run the sentiment-training section directly, or run all cells: annotation is skipped and the bundled `data/<dataset>/Labeled/` files are used.
- With `USE_PRECOMPUTED_LABELS = False`, run annotation and then sentiment training. Annotation reads the selected `Original/` files and writes both splits to `outputs/sentiment/<dataset>/`; training uses those outputs automatically.
- Start Jupyter from the LEMC checkout, or set `REPO_ROOT` to its absolute path. For annotation, put a compatible trained detector (configuration, saved tokenizer, and full `pytorch_model.bin`) in `checkpoints/metaphor/`, or update `MODEL_PATH`.

The SST2 evaluation partition is **validation**, including the bundled file named `test_sst2_metaphor.csv`. Rerun configuration after changing the selected dataset or annotation settings.


In [ ]:
from pathlib import Path
import pandas as pd

# Select one dataset: "Book", "IMDB", or "SST2".
DATASET_NAME = "Book"

# True: train from the repository's Labeled files and skip annotation.
# False: regenerate both splits, then train from outputs/sentiment/<dataset>/.
USE_PRECOMPUTED_LABELS = True

def find_repository_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "Metaphor_Sentiment.ipynb").is_file() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError(
        "Run this notebook from the LEMC checkout, or replace the REPO_ROOT "
        "assignment with Path('/absolute/path/to/LEMC')."
    )

REPO_ROOT = find_repository_root()
MODEL_PATH = REPO_ROOT / "checkpoints" / "metaphor"
CONCRETENESS_PATH = REPO_ROOT / "resources" / "Concreteness ratings.xlsx"
OUTPUT_ROOT = REPO_ROOT / "outputs" / "sentiment"

# Optional annotation controls. The default range processes each complete split.
START_INDEX = 0
END_INDEX = None  # Exclusive; use an integer to annotate a selected row range.
MINI_CHUNK_SIZE = 50
ANNOTATION_BATCH_SIZE = 512  # Reduce this if GPU memory is insufficient.

DATASET_CONFIGS = {
    "Book": {
        "original_train": "train.csv", "original_eval": "test.csv",
        "text_column": "text", "stem": "book", "evaluation_split": "test",
    },
    "IMDB": {
        "original_train": "train_imdb_with_metaphor_pos.csv",
        "original_eval": "test_imdb_with_metaphor_pos.csv",
        "text_column": "review", "stem": "imdb", "evaluation_split": "test",
    },
    "SST2": {
        "original_train": "train_sst2_with_metaphor_pos.csv",
        "original_eval": "val_sst2_with_metaphor_pos.csv",
        "text_column": "sentence", "stem": "sst2", "evaluation_split": "validation",
    },
}

def get_dataset_paths(dataset_name):
    if dataset_name not in DATASET_CONFIGS:
        raise ValueError(f"Unknown dataset {dataset_name!r}. Choose Book, IMDB, or SST2.")
    config = DATASET_CONFIGS[dataset_name]
    data_dir = Path(REPO_ROOT) / "data" / dataset_name
    output_dir = Path(OUTPUT_ROOT) / dataset_name
    stem = config["stem"]
    eval_prefix = "val" if config["evaluation_split"] == "validation" else "test"
    if not isinstance(START_INDEX, int) or START_INDEX < 0:
        raise ValueError("START_INDEX must be a non-negative integer.")
    if END_INDEX is not None and (not isinstance(END_INDEX, int) or END_INDEX <= START_INDEX):
        raise ValueError("END_INDEX must be None or an integer greater than START_INDEX.")
    suffix = "" if START_INDEX == 0 and END_INDEX is None else f"_{START_INDEX}_{END_INDEX if END_INDEX is not None else 'end'}"
    return {
        "name": dataset_name,
        "text_column": config["text_column"],
        "evaluation_split": config["evaluation_split"],
        "original_train": data_dir / "Original" / config["original_train"],
        "original_eval": data_dir / "Original" / config["original_eval"],
        "labeled_train": data_dir / "Labeled" / f"train_{stem}_metaphor.csv",
        # The bundled SST2 file named test_sst2_metaphor.csv is the validation split.
        "labeled_eval": data_dir / "Labeled" / f"test_{stem}_metaphor.csv",
        "generated_train": output_dir / f"train_{stem}_metaphor{suffix}.csv",
        "generated_eval": output_dir / f"{eval_prefix}_{stem}_metaphor{suffix}.csv",
    }

def require_csv(path):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"CSV file not found: {path}")
    with path.open("rb") as stream:
        if stream.read(128).startswith(b"version https://git-lfs.github.com/spec/v1"):
            raise ValueError(
                f"{path} is a Git LFS pointer, not the dataset. Download the full file "
                "with Git LFS (git lfs pull) before running this notebook."
            )
    return path

def load_annotation_frame(path, text_column):
    frame = pd.read_csv(require_csv(path))
    missing = {text_column, "label"} - set(frame.columns)
    if missing:
        raise ValueError(f"{path} is missing required columns: {sorted(missing)}")
    frame = frame.dropna(subset=[text_column]).copy()
    frame["text"] = frame[text_column].astype(str)
    frame = frame.loc[frame["text"].str.strip().ne(""), ["text", "label"]].reset_index(drop=True)
    if frame.empty:
        raise ValueError(f"No non-empty review text found in {path}")
    if not frame["label"].isin([0, 1]).all():
        raise ValueError(f"Expected binary sentiment labels (0/1) in {path}")
    return frame

def get_training_paths(dataset_name):
    paths = get_dataset_paths(dataset_name)
    source = "labeled" if USE_PRECOMPUTED_LABELS else "generated"
    train_path = require_csv(paths[f"{source}_train"])
    eval_path = require_csv(paths[f"{source}_eval"])
    return train_path, eval_path, paths["evaluation_split"]

selected_paths = get_dataset_paths(DATASET_NAME)
print(f"Dataset: {DATASET_NAME} | Evaluation partition: {selected_paths['evaluation_split']}")
print(f"Repository: {REPO_ROOT}")
print("Annotation mode: " + ("use bundled Labeled files" if USE_PRECOMPUTED_LABELS else "generate new annotations"))


# Predict and save metaphor_features (Train and Test Sets)

Optional when using the bundled `Labeled/` data. To regenerate annotations, set `USE_PRECOMPUTED_LABELS = False` above and run this cell with the trained detector and `en_core_web_sm` available. The code processes the train and evaluation splits, using spaCy tokenization, POS filtering, cached WordNet features, context windows, and mixed precision on CUDA.

Each review contributes at most its first 200 spaCy tokens; only NOUN, VERB, ADJ, and ADV receive detector predictions, while other POS tags receive literal label `0`. The full review is preserved and `metaphor_words` records the exact token sequence aligned with `metaphor_labels`, `pos_tags`, and `w_indices`. Subword truncation can still remove a marked target; the supplied prediction code uses the first token as its fallback target mask.

By default, both full splits are processed. `START_INDEX` and `END_INDEX` select a range of non-empty rows (end exclusive); partial ranges get a filename suffix. Adjust `ANNOTATION_BATCH_SIZE` if needed for GPU memory. Re-running a completed range replaces its generated output; an interrupted run leaves a `.partial.csv` file.


In [ ]:
if "USE_PRECOMPUTED_LABELS" not in globals():
    raise RuntimeError("Run the configuration cell first.")

if USE_PRECOMPUTED_LABELS:
    print("Skipping annotation: sentiment training will use the bundled Labeled files.")
else:
    import os
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset, DataLoader
    from transformers import RobertaTokenizer, RobertaConfig, RobertaModel
    import pandas as pd
    import nltk
    from nltk.corpus import wordnet as wn
    from collections import defaultdict
    import numpy as np
    from tqdm import tqdm
    from types import SimpleNamespace
    import spacy
    import string
    import gc
    from functools import lru_cache

    # --- 1. Initial setup and resource loading ---
    try:
        nltk.data.find('corpora/wordnet')
    except LookupError:
        nltk.download('wordnet', quiet=True)
        nltk.download('punkt', quiet=True)

    try:
        nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])
        print("spaCy model loaded successfully.")
    except OSError:
        print("Error: Run 'python -m spacy download en_core_web_sm' to download the model.")
        raise RuntimeError("Install the en_core_web_sm spaCy model before annotating reviews.")

    pos_to_id = defaultdict(lambda: len(pos_to_id))
    id_to_pos = {}
    common_pos_tags = ["NOUN", "VERB", "ADJ", "ADV", "PRON", "DET", "ADP", "NUM", "CONJ", "INTJ", "PART", "SCONJ", "SYM", "X", ".", "AUX", "PROPN", "PUNCT", "SPACE", "CCONJ"]
    for tag in common_pos_tags:
        pos_to_id[tag]

    spacy_to_custom = {
        'ADJ': 'ADJ', 'ADP': 'ADP', 'ADV': 'ADV', 'AUX': 'AUX', 'CCONJ': 'CCONJ',
        'DET': 'DET', 'INTJ': 'INTJ', 'NOUN': 'NOUN', 'NUM': 'NUM', 'PART': 'PART',
        'PRON': 'PRON', 'PROPN': 'PROPN', 'PUNCT': 'PUNCT', 'SCONJ': 'SCONJ',
        'SYM': 'SYM', 'VERB': 'VERB', 'X': 'X', 'SPACE': 'SPACE', 'CONJ': 'CCONJ'
    }

    # --- Standalone caching function (optimization: faster WordNet lookups) ---
    @lru_cache(maxsize=100000)
    def cached_wn_distance(word1, word2):
        """Cache WordNet distance calculations between two words to avoid repeated computation."""
        synsets1 = wn.synsets(word1)
        synsets2 = wn.synsets(word2)

        if not synsets1 or not synsets2:
            return 10.0, 0.0 # Default values

        s1 = synsets1[0]
        s2 = synsets2[0]

        dist = 10.0
        depth = 0.0

        # Path Distance
        try:
            d = s1.shortest_path_distance(s2)
            if d is not None: dist = d
        except: pass

        # LCH Depth
        try:
            lch = s1.lowest_common_hypernyms(s2)
            if lch: depth = lch[0].max_depth()
        except: pass

        return dist, depth

    class ConcretenessScorer:
        def __init__(self, file_path, dummy=False):
            self.dummy = dummy
            print(f"Loading concreteness ratings: {file_path}")
            try:
                df = pd.read_excel(file_path)
                self.concreteness_dict = pd.Series(df['Conc.M'].values, index=df['Word'].str.lower()).to_dict()
            except Exception as e:
                print(f"Failed to load concreteness ratings: {e}")
                self.concreteness_dict = {}

        def get_score(self, word):
            word = word.strip(string.punctuation)
            return self.concreteness_dict.get(str(word).lower(), 3.0)

        def get_wordnet_semantic_features(self, target_word, context_words):
            if self.dummy: return [1.0, 5.0, 0.0]

            path_dists = []
            lch_depths = []

            for cw in context_words:
                # Call the cached calculation function.
                dist, depth = cached_wn_distance(target_word, cw)
                path_dists.append(dist)
                if depth > 0: lch_depths.append(depth)

            avg_path = np.mean(path_dists) if path_dists else 0.0
            avg_lch = np.mean(lch_depths) if lch_depths else 0.0
            domain_diff = 1.0 if avg_path > 5.0 else 0.0

            return [avg_path, avg_lch, domain_diff]

    # --- 2. Model definition ---
    class MetaphorRoberta(nn.Module):
        def __init__(self, config, num_labels=2, num_pos_tags=len(pos_to_id), pos_embedding_dim=64, num_concreteness_features=6):
            super(MetaphorRoberta, self).__init__()
            self.num_labels = num_labels
            self.config = config
            self.encoder = RobertaModel(config=config)
            self.args = SimpleNamespace(drop_ratio=0.1, classifier_hidden=128, small_mean=True)
            self.dropout = nn.Dropout(self.args.drop_ratio)
            self.pos_embedding = nn.Embedding(num_embeddings=num_pos_tags, embedding_dim=pos_embedding_dim)
            self.self_attention = nn.MultiheadAttention(embed_dim=self.config.hidden_size, num_heads=8, dropout=self.args.drop_ratio, batch_first=True)

            input_dim_for_target_with_pos = self.config.hidden_size + pos_embedding_dim
            self.SPV_linear = nn.Linear(self.config.hidden_size + input_dim_for_target_with_pos, self.args.classifier_hidden)
            self.MIP_linear = nn.Linear(self.config.hidden_size + input_dim_for_target_with_pos, self.args.classifier_hidden)

            gate_input_dim = self.config.hidden_size + input_dim_for_target_with_pos
            self.gate_mlp = nn.Sequential(
                nn.Linear(gate_input_dim, self.args.classifier_hidden),
                nn.ReLU(),
                nn.Dropout(self.args.drop_ratio),
                nn.Linear(self.args.classifier_hidden, 2)
            )

            linguistic_feature_dim = 16
            self.linguistic_mlp = nn.Sequential(
                nn.Linear(num_concreteness_features, linguistic_feature_dim * 2),
                nn.ReLU(),
                nn.Dropout(0.1),
                nn.Linear(linguistic_feature_dim * 2, linguistic_feature_dim)
            )

            self.classifier = nn.Linear(self.args.classifier_hidden + linguistic_feature_dim, num_labels)
            self.logsoftmax = nn.LogSoftmax(dim=1)
            self._init_weights()

        def _init_weights(self):
            pass

        def forward(self, input_ids, input_ids_2, target_mask, target_mask_2, attention_mask, attention_mask_2, pos_ids=None, concreteness_features=None):
            outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
            sequence_output = outputs.last_hidden_state
            attn_output, _ = self.self_attention(sequence_output, sequence_output, sequence_output, key_padding_mask=~attention_mask.bool())
            sequence_output = attn_output
            cls_output = sequence_output[:, 0, :]

            target_output = sequence_output * target_mask.unsqueeze(2)
            target_output = self.dropout(target_output)
            target_output = target_output.sum(dim=1) / (target_mask.sum(-1, keepdim=True) + 1e-10)

            pos_embedding = self.pos_embedding(pos_ids)
            target_output_with_pos = torch.cat([target_output, pos_embedding], dim=1)

            outputs_2 = self.encoder(input_ids=input_ids_2, attention_mask=attention_mask_2)
            sequence_output_2 = outputs_2.last_hidden_state
            target_output_2 = sequence_output_2 * target_mask_2.unsqueeze(2)
            target_output_2 = self.dropout(target_output_2)
            target_output_2 = target_output_2.sum(dim=1) / (target_mask_2.sum(-1, keepdim=True) + 1e-10)

            SPV_hidden = self.SPV_linear(torch.cat([cls_output, target_output_with_pos], dim=1))
            MIP_hidden = self.MIP_linear(torch.cat([target_output_2, target_output_with_pos], dim=1))

            gate_input = torch.cat([cls_output, target_output_with_pos], dim=1)
            gate_weights = torch.sigmoid(self.gate_mlp(gate_input))
            SPV_weight, MIP_weight = gate_weights[:, 0:1], gate_weights[:, 1:2]
            combined_hidden = SPV_weight * SPV_hidden + MIP_weight * MIP_hidden

            linguistic_embedding = self.linguistic_mlp(concreteness_features)
            final_features = torch.cat([combined_hidden, linguistic_embedding], dim=1)

            logits = self.classifier(self.dropout(final_features))
            logits = self.logsoftmax(logits)
            return {'logits': logits}

    # --- 3. Dataset class ---
    class FlattenedMetaphorDataset(Dataset):
        def __init__(self, tasks, tokenizer, max_len=128):
            self.tasks = tasks
            self.tokenizer = tokenizer
            self.max_len = max_len

        def __len__(self):
            return len(self.tasks)

        def __getitem__(self, idx):
            item = self.tasks[idx]
            # Note: input_str already contains a short, windowed sentence and can be encoded directly.
            modified_sentence = item['input_str']
            target_word = item['target_word']

            encoding = self.tokenizer(modified_sentence, add_special_tokens=True, max_length=self.max_len, padding='max_length', truncation=True, return_tensors='pt')
            input_ids = encoding['input_ids'].flatten()

            target_mask = torch.zeros(self.max_len, dtype=torch.float)
            target_tokens = self.tokenizer.encode(f"[TARGET]{target_word}[/TARGET]", add_special_tokens=False)
            seq_len = len(target_tokens)
            found = False
            for i in range(len(input_ids) - seq_len + 1):
                if torch.equal(input_ids[i:i+seq_len], torch.tensor(target_tokens)):
                    target_mask[i:i+seq_len] = 1.0
                    found = True
                    break
            if not found: target_mask[0] = 1.0

            target_sentence_2 = target_word if target_word else "[PAD]"
            encoding_2 = self.tokenizer(target_sentence_2, add_special_tokens=True, max_length=self.max_len, padding='max_length', truncation=True, return_tensors='pt')
            input_ids_2 = encoding_2['input_ids'].flatten()
            target_mask_2 = torch.zeros(self.max_len, dtype=torch.float)
            target_tokens_2 = self.tokenizer.encode(target_word, add_special_tokens=False)
            seq_len_2 = len(target_tokens_2)
            for i in range(len(input_ids_2) - seq_len_2 + 1):
                if torch.equal(input_ids_2[i:i+seq_len_2], torch.tensor(target_tokens_2)):
                    target_mask_2[i:i+seq_len_2] = 1.0
                    break

            return {
                'input_ids': input_ids,
                'input_ids_2': input_ids_2,
                'target_mask': target_mask,
                'target_mask_2': target_mask_2,
                'attention_mask': encoding['attention_mask'].flatten(),
                'attention_mask_2': encoding_2['attention_mask'].flatten(),
                'pos_ids': torch.tensor(item['pos_id'], dtype=torch.long),
                'concreteness_features': torch.tensor(item['features'], dtype=torch.float)
            }

    # --- 4. Optimized processing (low RAM usage and faster inference) ---
    def process_and_predict_chunk(df_chunk, tokenizer, model, scorer, batch_size, device):
        """
        Review annotation with truncation, POS filtering, context windows, and mixed-precision inference.
        """
        # Increase the spaCy batch size.
        docs = list(nlp.pipe(df_chunk['text'], batch_size=50, disable=['parser', 'ner']))

        chunk_tasks = []
        chunk_results_map = []
        reviews_info = []

        # Optimization 1: Define the content-word POS tags to predict (POS filtering).
        # Skip function words such as pronouns (PRON), adpositions (ADP), and determiners (DET).
        TARGET_POS = {'NOUN', 'VERB', 'ADJ', 'ADV'}

        # Optimization 2: Set the truncation length for long review texts.
        ANALYSIS_LIMIT = 200

        for local_idx, (orig_idx, row) in enumerate(df_chunk.iterrows()):
            reviews_info.append({
                'review_idx': orig_idx,
                'text': row['text'],
                'label': row['label'],
                'pred_labels_map': {}
            })

            doc = docs[local_idx]

            # Apply optimization 2: Truncate long reviews to the first 200 tokens.
            if len(doc) > ANALYSIS_LIMIT:
                doc = doc[:ANALYSIS_LIMIT]

            words = [token.text for token in doc]
            pos_tags = [spacy_to_custom.get(token.pos_, 'NOUN') for token in doc]
            reviews_info[-1]['words'] = words
            reviews_info[-1]['pos_tags'] = pos_tags

            for w_idx, token in enumerate(doc):
                target_word = token.text
                current_pos = pos_tags[w_idx]

                # Apply optimization 1: Skip non-content words and label them as 0 (literal).
                if current_pos not in TARGET_POS:
                    reviews_info[local_idx]['pred_labels_map'][w_idx] = 0
                    continue

                # Optimization 3: Use a dynamic context window.
                # Reduce long-text truncation; subword tokenization may still remove the target.
                # Build a short window around target_word using up to 60 words on either side for the encoder.
                window_size = 60
                start = max(0, w_idx - window_size)
                end = min(len(words), w_idx + window_size)

                # Get context words for calculating WordNet features.
                context_words = words[start:w_idx] + words[w_idx+1:end]

                # Calculate features using cached lookups.
                t_conc = scorer.get_score(target_word)
                c_conc = 3.0
                if context_words:
                    c_scores = [scorer.get_score(w) for w in context_words]
                    c_conc = sum(c_scores) / len(c_scores)

                wn_feats = scorer.get_wordnet_semantic_features(target_word, context_words)
                features = [t_conc, c_conc, abs(t_conc - c_conc)] + wn_feats
                pos_id = pos_to_id[current_pos]

                # Build the input string (key correction).
                # Limit the word context; the tokenizer still truncates to 128 subword tokens.
                window_words = words[start:end].copy()
                # Recalculate the target position relative to the window.
                relative_target_idx = w_idx - start
                window_words[relative_target_idx] = f"[TARGET]{target_word}[/TARGET]"

                chunk_tasks.append({
                    'input_str': " ".join(window_words), # This is a short sentence.
                    'target_word': target_word,
                    'pos_id': pos_id,
                    'features': features
                })
                chunk_results_map.append((local_idx, w_idx))

        if not chunk_tasks:
            # Return immediately if this batch contains only skipped words.
            return _format_results(reviews_info)

        # 2. Prediction
        dataset = FlattenedMetaphorDataset(chunk_tasks, tokenizer)
        dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)

        all_preds = []

        # Optimization 4: Use automatic mixed precision (AMP) to accelerate GPU inference.
        with torch.no_grad():
            for batch in dataloader:
                input_ids = batch['input_ids'].to(device)
                input_ids_2 = batch['input_ids_2'].to(device)
                target_mask = batch['target_mask'].to(device)
                target_mask_2 = batch['target_mask_2'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                attention_mask_2 = batch['attention_mask_2'].to(device)
                pos_ids = batch['pos_ids'].to(device)
                concreteness_features = batch['concreteness_features'].to(device)

                # Enable the mixed-precision context.
                with torch.amp.autocast(device_type='cuda', enabled=(device.type == 'cuda')):
                    outputs = model(
                        input_ids=input_ids, input_ids_2=input_ids_2,
                        target_mask=target_mask, target_mask_2=target_mask_2,
                        attention_mask=attention_mask, attention_mask_2=attention_mask_2,
                        pos_ids=pos_ids, concreteness_features=concreteness_features
                    )
                    preds = torch.argmax(outputs['logits'], dim=1).cpu().tolist()
                    all_preds.extend(preds)

        # 3. Format the results
        for i, pred in enumerate(all_preds):
            local_idx, w_idx = chunk_results_map[i]
            reviews_info[local_idx]['pred_labels_map'][w_idx] = pred

        return _format_results(reviews_info)

    def _format_results(reviews_info):
        formatted_results = []
        for info in reviews_info:
            word_count = len(info['words'])
            # get(i, 0) ensures skipped words default to 0.
            labels_list = [info['pred_labels_map'].get(i, 0) for i in range(word_count)]
            formatted_results.append({
                'review': info['text'],
                'label': info['label'],
                'metaphor_words': str(info['words']),
                'metaphor_labels': str(labels_list),
                'pos_tags': str(info['pos_tags']),
                'w_indices': str(list(range(word_count)))
            })
        return formatted_results

    # --- Main function ---

    def main():
        paths = get_dataset_paths(DATASET_NAME)
        checkpoint_dir = Path(MODEL_PATH)
        for required in (checkpoint_dir / "config.json", checkpoint_dir / "pytorch_model.bin", Path(CONCRETENESS_PATH)):
            if not required.is_file():
                raise FileNotFoundError(
                    f"Required annotation resource not found: {required}. Train the detector "
                    "and set MODEL_PATH, or use USE_PRECOMPUTED_LABELS=True."
                )
        if MINI_CHUNK_SIZE < 1 or ANNOTATION_BATCH_SIZE < 1:
            raise ValueError("MINI_CHUNK_SIZE and ANNOTATION_BATCH_SIZE must be positive.")
        for split in ("train", "eval"):
            require_csv(paths[f"original_{split}"])

        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"Dataset: {DATASET_NAME} | Device: {device} | Range: {START_INDEX} -> {END_INDEX}")
        print("Loading model...")
        tokenizer = RobertaTokenizer.from_pretrained(str(checkpoint_dir), local_files_only=True)
        markers = {'[TARGET]', '[/TARGET]', '[LITERAL]'}
        if not markers.issubset(tokenizer.get_vocab()):
            raise ValueError("Use the trained detector's saved tokenizer, including its target/literal markers.")
        config = RobertaConfig.from_pretrained(str(checkpoint_dir), local_files_only=True)
        if config.vocab_size != len(tokenizer):
            raise ValueError("The checkpoint and tokenizer vocabulary sizes differ. Use matching saved files.")
        model = MetaphorRoberta(config, num_pos_tags=len(pos_to_id))
        # Require all custom detector layers; incompatible weights must not be silently ignored.
        state_dict = torch.load(checkpoint_dir / "pytorch_model.bin", map_location="cpu", weights_only=True)
        model.load_state_dict(state_dict, strict=True)
        del state_dict
        model.to(device)
        model.eval()
        scorer = ConcretenessScorer(CONCRETENESS_PATH)
        if not scorer.concreteness_dict:
            raise ValueError("No concreteness ratings were loaded. Check the workbook's Word and Conc.M columns.")

        for split in ("train", "eval"):
            file_path = paths[f"original_{split}"]
            out_path = paths[f"generated_{split}"]
            split_name = "train" if split == "train" else paths["evaluation_split"]
            print(f"\n>>> Processing {split_name}: {file_path}")
            df_full = load_annotation_frame(file_path, paths["text_column"])
            if START_INDEX >= len(df_full):
                raise ValueError(f"START_INDEX={START_INDEX} exceeds the {split_name} split length ({len(df_full)}).")
            real_end = len(df_full) if END_INDEX is None else min(END_INDEX, len(df_full))
            df_target = df_full.iloc[START_INDEX:real_end]
            print(f"Rows in the current task: {len(df_target)} (rows {START_INDEX} to {real_end}, exclusive)")
            out_path.parent.mkdir(parents=True, exist_ok=True)

            # Only publish a complete split. An interrupted run leaves a .partial.csv file.
            partial_path = out_path.with_suffix(".partial.csv")
            with tqdm(total=len(df_target), desc=f"Annotating {split_name}") as pbar:
                for i in range(0, len(df_target), MINI_CHUNK_SIZE):
                    df_mini = df_target.iloc[i:i + MINI_CHUNK_SIZE]
                    rows_in_chunk = len(df_mini)
                    results = process_and_predict_chunk(
                        df_mini, tokenizer, model, scorer,
                        batch_size=ANNOTATION_BATCH_SIZE, device=device,
                    )
                    result_df = pd.DataFrame(results)
                    result_df.to_csv(partial_path, mode='w' if i == 0 else 'a', header=(i == 0), index=False)
                    del df_mini, results, result_df
                    gc.collect()
                    pbar.update(rows_in_chunk)
            partial_path.replace(out_path)
            print(f"Saved: {out_path}")

    if __name__ == "__main__":
        main()


# Sentiment Analysis using metaphor_features

Run configuration first. This section automatically selects either the bundled annotations or the newly generated files for `DATASET_NAME`. It aligns stored `metaphor_words` to RoBERTa subwords. For bundled files without that column, it reconstructs spaCy English tokens, preserving whitespace tokens and the IMDb 200-token limit, and checks label counts.

Training uses `roberta-large` with the existing concatenation/projection model. Edit the model, batch size, epochs, and learning rate in `main()` below. Evaluation uses the selected test split for Book/IMDB and validation split for SST2. Sentiment checkpoint saving remains disabled in the commented saving line.


In [ ]:
if "get_training_paths" not in globals():
    raise RuntimeError("Run the configuration cell first.")

import os
import torch
import torch.nn as nn
import pandas as pd
import ast
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, RobertaForSequenceClassification, RobertaConfig
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tqdm import tqdm

# ==========================================
# 1. Metaphor Fusion Model without Attention (Concatenation Baseline)
# ==========================================
class RobertaMetaphorNoAttention(RobertaForSequenceClassification):
    def __init__(self, config):
        super().__init__(config)
        # 1. Metaphor Feature Embedding
        self.metaphor_embedding = nn.Embedding(2, config.hidden_size)
        self.dropout = nn.Dropout(0.1)

        # 2. Projection Layer (Replaces Attention)
        # Input dimension is text_hidden + meta_hidden (i.e. 2 * hidden_size)
        # Output dimension maps back to hidden_size
        self.fusion_projection = nn.Sequential(
            nn.Linear(config.hidden_size * 2, config.hidden_size),
            nn.Tanh(),
            nn.Dropout(0.1)
        )

        # Initialize custom weights
        self._init_custom_weights()

    def _init_custom_weights(self):
        for module in self.fusion_projection:
            if isinstance(module, nn.Linear):
                module.weight.data.normal_(mean=0.0, std=self.config.initializer_range)
                if module.bias is not None:
                    module.bias.data.zero_()

    def forward(self, input_ids, attention_mask=None, metaphor_features=None, labels=None):
        if metaphor_features is None:
            raise ValueError("Model requires 'metaphor_features' input.")

        # Text Features: (Batch, Seq_Len, Hidden)
        outputs = self.roberta(input_ids, attention_mask=attention_mask)
        sequence_output = outputs[0]

        # Metaphor Features: (Batch, Seq_Len, Hidden)
        metaphor_embeds = self.metaphor_embedding(metaphor_features)
        metaphor_embeds = self.dropout(metaphor_embeds)

        # --- Core Modification: Direct concatenation and fusion without Attention weights ---
        # 1. Concatenation
        combined_features = torch.cat((sequence_output, metaphor_embeds), dim=-1) # (Batch, Seq, 2*Hidden)

        # 2. Linear Projection Fusion
        fused_output = self.fusion_projection(combined_features) # (Batch, Seq, Hidden)

        # --- Classification Logic ---
        # Simple Mean Pooling
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(fused_output.size()).float()
        sum_embeddings = torch.sum(fused_output * input_mask_expanded, 1)
        sum_mask = input_mask_expanded.sum(1)
        sum_mask = torch.clamp(sum_mask, min=1e-9)
        pooled_output = sum_embeddings / sum_mask

        # Classifier
        logits = self.classifier(pooled_output.unsqueeze(1)) # Adapted to the input format of RobertaClassificationHead
        # Alternatively directly: logits = self.classifier.out_proj(self.classifier.dense(pooled_output))

        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))

        return {'loss': loss, 'logits': logits}

# ==========================================
# 2. Data Processing and Token Alignment
# ==========================================
class MetaphorSentimentDataset(Dataset):
    def __init__(self, csv_path, tokenizer, max_len=128, allow_truncated_reviews=False):
        print(f"Loading data: {csv_path}")
        self.data = pd.read_csv(require_csv(csv_path))
        if 'review' not in self.data.columns:
            for alias in ('sentence', 'text'):
                if alias in self.data.columns:
                    self.data.rename(columns={alias: 'review'}, inplace=True)
                    break
        if 'label' not in self.data.columns and 'sentiment' in self.data.columns:
            self.data.rename(columns={'sentiment': 'label'}, inplace=True)
        missing = {'review', 'label', 'metaphor_labels'} - set(self.data.columns)
        if missing:
            raise ValueError(f"{csv_path} is missing required columns: {sorted(missing)}")
        if self.data.empty or self.data['review'].isna().any():
            raise ValueError(f"Expected non-empty labeled review data in {csv_path}")
        if not self.data['label'].isin([0, 1]).all():
            raise ValueError(f"Expected binary sentiment labels (0/1) in {csv_path}")
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.csv_path = csv_path
        self.allow_truncated_reviews = allow_truncated_reviews
        # Existing Labeled CSVs store spaCy-token labels without the original token list.
        # Only tokenization is needed here; no POS model or detector checkpoint is loaded.
        self.word_tokenizer = None
        if 'metaphor_words' not in self.data.columns:
            import spacy
            self.word_tokenizer = spacy.blank('en').tokenizer

    def __len__(self):
        return len(self.data)

    @staticmethod
    def parse_list(value, column, idx):
        try:
            parsed = ast.literal_eval(value) if isinstance(value, str) else value
        except (ValueError, SyntaxError) as exc:
            raise ValueError(f"Row {idx}: invalid {column} list.") from exc
        if not isinstance(parsed, list):
            raise ValueError(f"Row {idx}: {column} must be a list.")
        return parsed

    def review_words_and_labels(self, row, idx):
        word_labels = self.parse_list(row['metaphor_labels'], 'metaphor_labels', idx)
        if not word_labels or any(label not in (0, 1) for label in word_labels):
            raise ValueError(f"Row {idx}: metaphor_labels must contain binary labels (0/1).")
        if 'metaphor_words' in self.data.columns:
            words = self.parse_list(row['metaphor_words'], 'metaphor_words', idx)
            if any(not isinstance(word, str) or not word for word in words):
                raise ValueError(f"Row {idx}: metaphor_words must contain non-empty strings.")
        else:
            words = [token.text for token in self.word_tokenizer(str(row['review']))]
            # The bundled IMDb annotations cover at most the first 200 spaCy tokens.
            if self.allow_truncated_reviews and len(word_labels) == 200 and len(words) > 200:
                words = words[:200]
        if len(words) != len(word_labels):
            raise ValueError(
                f"{self.csv_path}, row {idx}: {len(words)} tokens but {len(word_labels)} metaphor labels. "
                "Regenerate annotations with this notebook to save the exact metaphor_words."
            )
        return words, word_labels

    def align_labels(self, tokenized_inputs, word_labels):
        return [0 if word_idx is None else word_labels[word_idx]
                for word_idx in tokenized_inputs.word_ids()]

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        words, word_labels = self.review_words_and_labels(row, idx)
        encoding = self.tokenizer(
            words,
            is_split_into_words=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        aligned_metaphor_labels = self.align_labels(encoding, word_labels)
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'metaphor_features': torch.tensor(aligned_metaphor_labels, dtype=torch.long),
            'labels': torch.tensor(int(row['label']), dtype=torch.long),
        }


# ==========================================
# 3. Evaluation and Training Functions
# ==========================================
def evaluate_detailed(model, dataloader, device):
    model.eval()
    preds = []
    true_labels = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating", leave=False):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            metaphor_features = batch['metaphor_features'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                metaphor_features=metaphor_features
            )

            logits = outputs['logits']
            batch_preds = torch.argmax(logits, dim=1).cpu().tolist()
            preds.extend(batch_preds)
            true_labels.extend(labels.cpu().tolist())

    acc = accuracy_score(true_labels, preds)
    p = precision_score(true_labels, preds, pos_label=1)
    r = recall_score(true_labels, preds, pos_label=1)
    f1 = f1_score(true_labels, preds, pos_label=1)

    return acc, p, r, f1

def train_epoch(model, dataloader, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    progress = tqdm(dataloader, desc="Training", leave=False)

    for batch in progress:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        metaphor_features = batch['metaphor_features'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, metaphor_features=metaphor_features, labels=labels)
        loss = outputs['loss']
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        progress.set_postfix({'loss': loss.item()})

    return total_loss / len(dataloader)

# ==========================================
# 4. Main Pipeline
# ==========================================
def main():
    # Configuration
    TRAIN_FILE, EVAL_FILE, evaluation_split = get_training_paths(DATASET_NAME)
    MODEL_NAME = 'roberta-large'
    BATCH_SIZE = 32
    EPOCHS = 4
    LR = 1e-5

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Dataset: {DATASET_NAME} | Device: {device} | Evaluation: {evaluation_split}")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True, add_prefix_space=True)

    print("\n=== Experiment: Direct Embedding of Metaphor Features (No Attention) ===")

    # The bundled IMDb labels cover at most the first 200 spaCy tokens.
    allow_truncated = DATASET_NAME == "IMDB"
    train_ds = MetaphorSentimentDataset(TRAIN_FILE, tokenizer, allow_truncated_reviews=allow_truncated)
    eval_ds = MetaphorSentimentDataset(EVAL_FILE, tokenizer, allow_truncated_reviews=allow_truncated)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    eval_loader = DataLoader(eval_ds, batch_size=BATCH_SIZE)

    # Initialize Model (using the NoAttention version)
    config = RobertaConfig.from_pretrained(MODEL_NAME, num_labels=2)
    model = RobertaMetaphorNoAttention.from_pretrained(MODEL_NAME, config=config)
    model.to(device)

    optimizer = AdamW(model.parameters(), lr=LR)
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1*len(train_loader)*EPOCHS), num_training_steps=len(train_loader)*EPOCHS)

    best_f1 = 0
    for epoch in range(EPOCHS):
        print(f"\nEpoch {epoch+1}/{EPOCHS}")
        avg_loss = train_epoch(model, train_loader, optimizer, scheduler, device)
        print(f"  Train Loss: {avg_loss:.4f}")

        acc, prec, rec, f1 = evaluate_detailed(model, eval_loader, device)
        print(f"  {evaluation_split.title()} Metrics - Acc: {acc:.4f}, Prec: {prec:.4f}, Rec: {rec:.4f}, F1: {f1:.4f}")

        if f1 > best_f1:
            best_f1 = f1
            print(f"  >> New Best No-Attn Model! (F1: {best_f1:.4f})")
            # torch.save(model.state_dict(), 'no_attn_model.bin')

if __name__ == "__main__":
    main()